<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab06_injury_risk_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 6 — An injury-risk classifier, honestly
**Artificial Intelligence Applications in Sport · Week 6**

⏱️ About 80 minutes · 🧪 This notebook creates its own data — nothing can fail to download

> **First:** `File → Save a copy in Drive`. Cells marked ✏️ **EDIT ME** are the ones you change.


### What you will produce
A calibrated injury-risk model built the correct way from the start — and a **threshold recommendation with a
number of false alarms per season attached to it**, which is the only form in which such a recommendation is useful.

### The rule you carry from Week 5
1. Compute the trivial baseline before anything else.  2. No column written after the outcome.
3. Split by athlete, never by row.  4. A high number is a reason for suspicion.


---
## Block 1 — Setup


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, confusion_matrix)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
RNG = np.random.default_rng(6)
print('Setup complete.')


---
## Block 2 — The data

Weekly summaries for 30 athletes over 26 weeks. The label is **an injury occurring in the following week** —
note the direction: everything in the row is known *before* the outcome. That is what makes this a prediction
problem rather than a description.


In [ ]:
import numpy as np, pandas as pd

RNG = np.random.default_rng(6)
N_ATH, N_WEEKS, SESS = 45, 40, 6      # a squad plus academy, tracked across two seasons

ath = pd.DataFrame({
    'athlete_id':   [f'A{i:02d}' for i in range(N_ATH)],
    'age':          RNG.integers(17, 34, N_ATH),
    'body_mass_kg': np.round(RNG.normal(76, 7, N_ATH), 1),
})
ath['frailty'] = RNG.normal(0, 1, N_ATH)      # real risk, never measured

rows = []
for _, a in ath.iterrows():
    for wk_i in range(1, N_WEEKS + 1):
        for s in range(SESS):
            match = (s == SESS - 1)
            dur  = RNG.normal(95 if match else 78, 12)
            rpe  = np.clip(RNG.normal(8.2 if match else 5.9, 1.3), 1, 10)
            dist = np.clip(RNG.normal(9800 if match else 5400, 1100), 500, None)
            rows.append(dict(
                athlete_id=a.athlete_id, week=wk_i,
                duration_min=dur, rpe=rpe, total_distance_m=dist,
                high_speed_distance_m=np.clip(dist*RNG.normal(.09 if match else .05, .02), 0, None),
                accelerations=np.clip(RNG.normal(46 if match else 28, 9), 0, None),
                sleep_h=np.clip(RNG.normal(7.4, .9), 4, 10),
                wellness=np.clip(round(RNG.normal(3.6, .9)), 1, 5),
                soreness=np.clip(round(RNG.normal(2.4 + 1.0*match, .9)), 1, 5),
                age=a.age, body_mass_kg=a.body_mass_kg, frailty=a.frailty))

sess = pd.DataFrame(rows)
sess['session_load'] = sess.rpe * sess.duration_min

wk = (sess.groupby(['athlete_id','week'])
          .agg(load=('session_load','sum'), distance_km=('total_distance_m', lambda s: s.sum()/1000),
               hsd_m=('high_speed_distance_m','sum'), accels=('accelerations','sum'),
               sessions=('session_load','size'), sleep_h=('sleep_h','mean'),
               wellness=('wellness','mean'), soreness_mean=('soreness','mean'),
               soreness_max=('soreness','max'), age=('age','first'),
               body_mass_kg=('body_mass_kg','first'), frailty=('frailty','first'))
          .reset_index().sort_values(['athlete_id','week']))

# acute : chronic workload ratio, computed only from weeks already past
g = wk.groupby('athlete_id')['load']
wk['chronic_4wk'] = g.transform(lambda s: s.shift(1).rolling(4, min_periods=1).mean())
wk['acwr']        = (wk.load / wk.chronic_4wk).replace([np.inf, -np.inf], np.nan)
wk['load_change'] = g.transform(lambda s: s.diff())
wk = wk.dropna(subset=['acwr','load_change']).reset_index(drop=True)

# the truth: next-week injury depends on unmeasured frailty, a workload spike,
# accumulated soreness, short sleep, and high-speed exposure
hsd_z = (wk.hsd_m - wk.hsd_m.mean()) / wk.hsd_m.std()
lin = (-5.80
       + 0.30 * wk.frailty
       + 3.20 * (wk.acwr - 1)
       + 0.85 * wk.soreness_max
       - 0.60 * (wk.sleep_h - 7.4)
       + 0.75 * hsd_z)
wk['injury_next_week'] = RNG.binomial(1, 1/(1 + np.exp(-lin)))

FEATURES = ['load','distance_km','hsd_m','accels','sessions','sleep_h','wellness',
            'soreness_mean','soreness_max','age','body_mass_kg','acwr','load_change']
X, y, groups = wk[FEATURES].fillna(wk[FEATURES].median()), wk.injury_next_week, wk.athlete_id

print(f'{len(wk)} athlete-weeks · {groups.nunique()} athletes · injury rate {y.mean():.1%} ({y.sum()} injuries)')
wk[['athlete_id','week','load','acwr','soreness_max','sleep_h','injury_next_week']].head()


---
## Block 3 — The trivial baseline, first

Always. Before any model.


In [ ]:
print('Rule: "nobody will be injured next week"')
print(f'  Accuracy        : {1-y.mean():.1%}')
print(f'  Injuries caught : 0 of {y.sum()}')
print(f'  AUC             : 0.500')
print('\nAny model below must be judged against this, not against zero.')


---
## Block 4 — The model, split by athlete

`GroupKFold` guarantees that an athlete is never in the training fold and the test fold at the same time.
`cross_val_predict` gives us an out-of-fold probability for every athlete-week, so every prediction below comes
from a model that had never seen that athlete.


In [ ]:
clf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                             class_weight='balanced', random_state=0, n_jobs=-1)

proba = cross_val_predict(clf, X, y, groups=groups, cv=GroupKFold(n_splits=5),
                          method='predict_proba')[:, 1]

auc  = roc_auc_score(y, proba)
prauc = average_precision_score(y, proba)
print(f'ROC-AUC          : {auc:.3f}   (0.50 = coin flip)')
print(f'PR-AUC           : {prauc:.3f}   (baseline = the injury rate, {y.mean():.3f})')
print('\nPublished models evaluated this carefully typically land at 0.73-0.77.')


In [ ]:
fpr, tpr, _ = roc_curve(y, proba)
prec, rec, _ = precision_recall_curve(y, proba)
frac_pos, mean_pred = calibration_curve(y, proba, n_bins=8, strategy='quantile')

fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
axes[0].plot(fpr, tpr, color='#8A1C2B', lw=2); axes[0].plot([0,1],[0,1],'--',color='#999')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate')
axes[0].set_title(f'ROC  ·  AUC = {auc:.3f}')

axes[1].plot(rec, prec, color='#1B6470', lw=2)
axes[1].axhline(y.mean(), ls='--', color='#999')
axes[1].set_xlabel('Recall (injuries caught)'); axes[1].set_ylabel('Precision (flags that were right)')
axes[1].set_title(f'Precision-recall  ·  PR-AUC = {prauc:.3f}')

axes[2].plot(mean_pred, frac_pos, 'o-', color='#9A5714')
axes[2].plot([0, max(mean_pred)], [0, max(mean_pred)], '--', color='#999')
axes[2].set_xlabel('Predicted probability'); axes[2].set_ylabel('Observed rate')
axes[2].set_title('Calibration')
for a in axes: a.grid(alpha=.3)
plt.tight_layout(); plt.show()


**Read the three panels as three different questions.** ROC asks *can the model rank athletes?* Precision-recall
asks *of the athletes it flags, how many are really at risk?* — the honest question when injuries are rare.
Calibration asks *when it says 20%, does 20% of that group get injured?* A model can rank well and still be badly
calibrated, and a coach who is told “35% risk” is being told a number that must mean something.


---
## Block 5 — The threshold is a policy decision, not a statistical one

The model outputs a probability. **Somebody has to choose the cut-off**, and that choice says how many missed
injuries you will trade for how many unnecessary rest days. That somebody is you, not the algorithm.


In [ ]:
COST_MISSED_INJURY = 10   # ✏️ EDIT ME — how many false alarms is one missed injury worth?
COST_FALSE_ALARM   = 1
WEEKS_PER_SEASON   = 26
n_ath = groups.nunique()

rows = []
for t in np.arange(0.05, 0.61, 0.01):
    tn, fp, fn, tp = confusion_matrix(y, (proba >= t).astype(int), labels=[0,1]).ravel()
    rows.append(dict(threshold=round(t,2), caught=tp, missed=fn, false_alarms=fp,
                     sensitivity=tp/(tp+fn) if tp+fn else 0,
                     precision=tp/(tp+fp) if tp+fp else 0,
                     cost=COST_MISSED_INJURY*fn + COST_FALSE_ALARM*fp))

t_tbl = pd.DataFrame(rows)
best_t = t_tbl.loc[t_tbl.cost.idxmin()]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(t_tbl.threshold, t_tbl.sensitivity, color='#8A1C2B', lw=2, label='sensitivity (injuries caught)')
ax.plot(t_tbl.threshold, t_tbl.precision,  color='#1B6470', lw=2, label='precision (flags that were right)')
ax.axvline(best_t.threshold, ls='--', color='#9A5714',
           label=f'cost-minimising threshold = {best_t.threshold:.2f}')
ax.set_xlabel('Decision threshold'); ax.set_ylabel('Proportion')
ax.set_title(f'The trade-off at a cost ratio of {COST_MISSED_INJURY}:1'); ax.legend(frameon=False)
ax.grid(alpha=.3); plt.tight_layout(); plt.show()

fa_per_season = best_t.false_alarms / n_ath
print(f'At threshold {best_t.threshold:.2f}, over one season of {n_ath} athletes:')
print(f'  injuries caught        : {int(best_t.caught)} of {int(best_t.caught + best_t.missed)}')
print(f'  injuries missed        : {int(best_t.missed)}')
print(f'  false alarms           : {int(best_t.false_alarms)}  '
      f'= {fa_per_season:.1f} per athlete per season')
print(f'\nThat last number is the one your medical staff will actually argue about.')


---
## Block 6 — Break it 💥

Set `COST_MISSED_INJURY` above to `1` (a missed injury costs the same as a false alarm) and re-run.
The threshold jumps, the model stops flagging almost anyone, and it still reports the same AUC.

**The AUC did not change because the model did not change. Only the policy did.** Any vendor who quotes you an
accuracy figure without telling you their threshold has told you nothing.


---
## Block 7 — What is it using?


In [ ]:
clf.fit(X, y)
pi  = permutation_importance(clf, X, y, n_repeats=15, random_state=0, scoring='roc_auc')
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values()

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.barh(imp.index, imp.values, color='#1B6470')
ax.set_xlabel('Drop in AUC when this column is shuffled')
ax.set_title('Injury model — permutation importance'); ax.grid(axis='x', alpha=.3)
plt.tight_layout(); plt.show()

print('Write one sentence per top feature that a coach would accept.')
print('If you cannot, you cannot deploy this model — whatever its AUC.')


---
## Interpretation ✍️ **(graded)**

About 200 words:

1. **State your recommended threshold**, and the number of false alarms per athlete per season it implies.
2. Would your medical staff accept that number? What would you have to change if they would not?
3. The model was built by never letting an athlete appear in both the training and test folds. Explain to a coach,
   in one sentence with no jargon, why that mattered.
4. Name one thing this model cannot know that a physiotherapist watching the session would.

---

**Your answer:**

*(replace this line)*


---
## AI Use Log

Complete honestly — it is part of the mark. Write *none* if you used no AI tool.

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
### Submitting

1. `File → Save a copy in Drive` · 2. `Share → Anyone with the link → Viewer` · 3. Paste the link into ODTÜClass.
